<a href="https://colab.research.google.com/github/vishal-anand-max/DataScienceEssentials/blob/main/Vectorization_Dojo_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📜 The Math-to-NumPy Protocol (v2.2)
*The Universal Translation Layer (Patched for Deep Physics)*

***Objective:*** A structured framework to convert mathematical expressions into vectorized NumPy code.

### Step 1: The Dependency Check (The Stop Sign)
* **Objective:** Identify Recursion.
* **Look at:** RHS Indices.
* **Action:**
    * *Recursive* ($x_t$ depends on $x_{t-1}$): **STOP**. Use a Loop.
    * *Independent*: Proceed to Vectorization.

### Step 2: The Global Map (The LHS Lock)
* **Objective:** Define the Tensor Universe.
* **Logic:** Distinguish between indices that stay and indices that vanish.
    * **Free Indices (LHS):** These dictate the permanent axes of your final tensor.
    * **Summation Indices (RHS):** Temporary axes used for calculation, then reduced (summed).
* **Example:** $R_{ij} = \sum_k (\dots)$
    * $i \to$ Axis 0
    * $j \to$ Axis 1
    * $k \to$ Axis 2 (to be reduced)

### Step 3: The Shape Classifier (The Branching Path)
* **Objective:** Handle Iteration Limits & Access Patterns.
* **Look at:** The relationship between indices.
* **Action:**
    1.  **Rectangular (Standard):**
        * *Sign:* Sums over full ranges ($\sum_{i=1}^N$).
        * *Tool:* Standard Broadcasting or Slicing (`:`)
    2.  **Irregular (Triangular):**
        * *Sign:* Sum depends on position ($\sum_{j<i}$).
        * *Tool:* **Masking** (`A * np.tril(1)`) or `np.where`.
    3.  **Indirect (Scatter/Mesh):**
        * *Sign:* LHS index is a list/map (e.g., $A[\text{nodes}_i] \leftarrow \dots$).
        * *Logic:* Multiple inputs might map to the same output "bin" (Collisions).
        * *Tool:* **`np.add.at(Target, Indices, Values)`**. (Note: Standard assignment `=` will fail here).

### Step 4: Vector Representation
* **Policy:** Store data as Flat Arrays $(N,)$.
* **Constraint:** Only promote to Column Vectors $(N,1)$ immediately before a strict Matrix Dot Product (`@`).

### Step 5: The Alignment (Explicit Broadcasting)
* **Objective:** Mechanics over Magic.
* **Action:** Insert `None` (or `np.newaxis`) for missing indices to match the **Global Map**.
* **Target:** If Global Map is $(i, j)$ and term has $j$ only $\to$ `Term[None, :]`.
* **Shuffle** Apart from adding new axis using `Term[None,:]` you can also transpose and shuffle the axes to align the array with the global map
* **Prerequisite** To better appreciate this step, you should know the rules of broadcasting

# 🥋 Vectorization Dojo


## Challenge 1
### The Equation: Weighted Interaction Matrix
We need to compute matrix $E$, where each element is an interaction between vector $A$ and vector $B$, normalized by the sum of interactions.

$$E_{ij} = \frac{\exp(A_i \cdot B_j)}{\sum_{k} \exp(A_i \cdot B_k)}$$

**Variables:**
* $A$: Vector of size $N$
* $B$: Vector of size $M$

## Step 1: Dependency Check
**Question:** Is there recursion? Does $E_{ij}$ depend on $E_{i-1, j}$?

**Analysis:** No. All indices ($i, j, k$) are independent.
**Decision:** Proceed to Vectorization.

## Step 2: The Global Map
**Question:** What are the axes of our universe?

**Analysis:**
* **Free Indices (LHS):** $i, j$. These form the output shape.
* **Summation Indices (RHS):** $k$. This axis will be reduced.

**Mapping:**
* $i \to$ Axis 0
* $j \to$ Axis 1

In [ ]:
import numpy as np

# Setup Dummy Data
N = 5
M = 3
A = np.random.rand(N) # Shape (N,)
B = np.random.rand(M) # Shape (M,)

## Step 5: The Alignment & Solution

**Logic:**
1. **Align:** We need $A_i$ (Axis 0) and $B_j$ (Axis 1).
   - $A \to$ `A[:, None]`
   - $B \to$ `B[None, :]`
2. **Numerator:** Compute the interaction grid $(N, M)$.
3. **Denominator:** Compute the interaction grid using $B_k$, sum over $k$ (Axis 1), and **reshape** to maintain alignment.

In [ ]:
# 1. The Numerator
# Map: (i, j) -> (Axis 0, Axis 1)
interaction = A[:, None] * B[None, :]  # Broadcasts to (N, M)
numerator = np.exp(interaction)

# 2. The Denominator
# We sum over k (which sits in Axis 1 position for B)
# Trap: np.sum reduces dimension to (N,). We must keep it (N, 1) for division.
denominator_sum = np.sum(numerator, axis=1) # Result: (N,)
denominator = denominator_sum[:, None]      # Result: (N, 1)

# 3. Final Calculation
# (N, M) / (N, 1) -> Valid Broadcast
E = numerator / denominator

print(f"Shape of A: {A.shape}")
print(f"Shape of B: {B.shape}")
print(f"Shape of Result E: {E.shape}")

Shape of A: (5,)
Shape of B: (3,)
Shape of Result E: (5, 3)


## Challenge 2
### The Equation: Causal Cumulative Signal
We want to calculate a signal $Y$ at time $t$, which is the weighted sum of inputs $X$ from all *previous* times $k$ (up to and including $t$).

$$Y_t = \sum_{k=0}^{t} (W_{tk} \cdot X_k)$$

**Variables:**
* $W$: Weight matrix of shape $(N, N)$ (Time-dependent weights)
* $X$: Input signal vector of size $(N,)$
* $t, k$: Time indices from $0$ to $N-1$

### Protocol Analysis
* **Step 1 (Dependency Check):**
    * **Question:** Is there recursion?
    * **Answer:** No. $Y_t$ does not depend on the *result* of $Y_{t-1}$. It is an independent sum.
    * **Decision:** Vectorize.
* **Step 2 (The Global Map):**
    * **Free Indices:** $t$ (LHS).
    * **Summation Index:** $k$ (RHS).
    * **Map:** $t \to$ Axis 0, $k \to$ Axis 1.
* **Step 3 (Shape Classifier):**
    * **Limit:** $k=0$ to $t$ ($k \le t$).
    * **Type:** Irregular / Triangular.
    * **Action:** We must use **Masking** (`np.tril`) because we are vectorizing the loop over $t$.
* **Step 5 (Alignment - The "Axis Trap"):**
    * **Term:** $X_k$.
    * **Target:** $X$ must align with $k$ (Axis 1) to multiply against $W_{tk}$.
    * **Action:** Reshape $X$ to `(1, N)` using `X[None, :]`. (Note: `X[:, None]` would align to $t$, which is incorrect).

---

In [ ]:
import numpy as np
N =30
W =np.random.rand(N,N)
X =np.random.rand(N,)
T =W[:,:]*X[None,:]
T2 =np.tril(T)
Y =np.sum(T2,axis =1)
print(Y)

[0.56101492 0.39371597 0.63015652 0.64815842 1.26252972 1.95719294
 2.63495951 1.24018195 3.54107134 3.91439347 3.68519525 3.96084038
 3.69482879 5.45810953 4.15399675 4.80090873 5.05757397 6.85269755
 6.02821763 4.20722038 6.06812705 6.32742871 7.36774263 5.42421401
 6.87195824 8.30180117 8.39446353 8.12663293 8.69502018 8.46079946]


## Challenge 2.5 (The Retrial)
### The Equation: The Cumulative Field
We are calculating the total Field Strength $F$ at position $i$. The field at $i$ is the sum of influences from all *previous* positions $j$ (where $j \le i$), weighted by the mass $M$ at that position.

$$F_i = \sum_{j=0}^{i} (G_{ij} \cdot M_j)$$

**Variables:**
* $G$: Interaction Grid of shape `(N, N)` (Row $i$, Col $j$)
* $M$: Mass vector of shape `(N,)`
* $i, j$: Spatial indices from $0$ to $N-1$

### Protocol Analysis
* **Step 2 (The Global Map):**
    * **Free Index:** $i$ (Axis 0).
    * **Summation Index:** $j$ (Axis 1).
* **Step 3 (Shape Classifier):**
    * **Limit:** $j \le i$.
    * **Geometry:** Lower Triangular.
    * **Tool:** `np.tril` (Triangle Lower).
* **Step 5 (Alignment):**
    * **Term:** $M_j$.
    * **Target:** $j$ is Axis 1.
    * **Action:** Reshape $M$ to `(1, N)` using `M[None, :]`.

In [ ]:
import numpy as np
N=30
G =np.random.rand(N,N)
M =np.random.rand(N,)
T =G[:,:]*M[None,:]
T1 =np.tril(T)
F =np.sum(T1,axis=1)
print(F)

[0.5278943  1.08879248 0.28609584 1.14894167 0.98657627 1.69111298
 2.08289168 2.9674951  3.64820939 2.98143163 3.56872697 4.86807951
 3.64075533 3.65219412 3.76188515 5.09090575 4.46570552 6.39068729
 5.02750436 5.26773359 6.56069686 5.7474962  6.10253303 6.68723558
 7.45097915 8.74639484 8.51100085 6.68182216 6.59492617 8.83205622]


## Challenge 3 (The Final Boss)
### The Equation: Pairwise Particle Distances (3D)
We have two sets of particles in 3D space. We want to calculate the squared Euclidean distance between every particle in set $P$ and every particle in set $Q$.

$$D_{ij} = \sum_{k \in \{x,y,z\}} (P_{ik} - Q_{jk})^2$$

**Variables:**
* $P$: Matrix of shape `(N, 3)` (N particles, 3 coords).
* $Q$: Matrix of shape `(M, 3)` (M particles, 3 coords).
* $k$: The coordinate index (0=x, 1=y, 2=z).

### Protocol Analysis
* **Step 2 (The Global Map - High Dimensional):**
    * **Free Indices:** $i$ (Axis 0), $j$ (Axis 1).
    * **Summation Index:** $k$ (Axis 2).
    * **Note:** $k$ exists in the input but is summed out in the output.
* **Step 5 (Alignment):**
    * **Term $P_{ik}$:** Has axes $(i, k)$. Missing $j$ (Axis 1).
        * **Action:** `P[:, None, :]` $\to$ Shape `(N, 1, 3)`.
    * **Term $Q_{jk}$:** Has axes $(j, k)$. Missing $i$ (Axis 0).
        * **Action:** `Q[None, :, :]` $\to$ Shape `(1, M, 3)`.
* **Reduction:**
    * After computing the squared difference `(N, M, 3)`, we sum over $k$ (Axis 2).

In [ ]:
import numpy as np
N =30
M =20
P =np.random.rand(N,3)
Q =np.random.rand(M,3)
T =P[:,None,:]-Q[None,:,:]
T2 =T*T
D=np.sum(T2,axis=2)
print(D.shape)

(30, 20)


# 🧙‍♂️ The Ultimate Weapon: `np.einsum` (Einstein Summation)

While the Math-to-NumPy protocol (specifically explicit broadcasting with `None`) is robust and mathematically sound, NumPy's built-in operators like `np.dot` and `@` can feel inconsistent. They try to "guess" your intentions based on the shape of your arrays (1D vs 2D vs ND), which often leads to shape-mismatch nightmares.

**`np.einsum` is the antidote to that magic.** It relies on **Einstein Summation Convention**, which forces you to be explicit about exactly which axes are interacting and which are being reduced, bypassing the need for `None` or `np.newaxis` during multiplication.

### Why `einsum` is a Game Changer for Physics:
1. **Supreme Explicitness:** You name your dimensions using letters (e.g., `i, j, k`). You write the math exactly as it appears on paper.
2. **No Magic Axes:** You don't have to remember if `np.tensordot` needs `axis=0` or `axis=1`. The string dictates the layout.
3. **Replaces Multiple Functions:** It can do inner products, outer products, traces, transposes, and batched matrix multiplications just by changing the index string.
4. **Memory Optimization:** By adding the `optimize=True` flag, NumPy evaluates the most mathematically efficient path to contract multiple tensors, often skipping the creation of massive intermediate arrays.

### The Syntax:
`np.einsum('input1_indices, input2_indices -> output_indices', Array1, Array2)`

* **Free Indices:** Letters that appear on *both* sides of the `->`. These are kept in the final shape.
* **Summation Indices:** Letters that appear on the *left* but NOT on the *right*. These are summed out (reduced).

In [2]:
# Challenge 1: Weighted Interaction Matrix using einsum
import numpy as np

N = 5
M = 3
A = np.random.rand(N)
B = np.random.rand(M)

# Step 1: The Numerator (Outer Product)
# Protocol Map: A gets 'i', B gets 'j'. Output needs both 'i' and 'j'.
# No None/newaxis needed! einsum handles the alignment.
interaction = np.einsum('i, j -> ij', A, B)
numerator = np.exp(interaction)

# Step 2: The Denominator (Summation)
# We want to sum over 'j' (which acts as our 'k' in the formula).
# einsum reduces 'j' and leaves us with 'i'.
denominator_sum = np.einsum('ij -> i', numerator)

# Step 3: Final Division
# Note: einsum does not do division, so we still rely on your protocol
# to align the denominator for broadcasting: (N, M) / (N, 1)
E = numerator / denominator_sum[:, None]

print(f"Shape of Result E: {E.shape}")

Shape of Result E: (5, 3)


In [3]:
# Challenge 2.5: The Cumulative Field using einsum
import numpy as np

N = 30
G = np.random.rand(N, N)
M = np.random.rand(N,)

# Note on Protocol: einsum cannot evaluate conditions like "j <= i".
# We MUST still use np.tril to apply the mask to the interaction grid G first.
G_masked = np.tril(G)

# Using einsum for the Weighted Sum:
# G_masked has indices 'ij'. M has index 'j'.
# We multiply them and sum over 'j', leaving only 'i'.
# This entirely replaces G[:, :] * M[None, :] AND np.sum(..., axis=1)
F = np.einsum('ij, j -> i', G_masked, M)

print(f"Shape of Cumulative Field F: {F.shape}")

Shape of Cumulative Field F: (30,)


In [4]:
# Challenge 3: Pairwise Particle Distances (3D) using einsum
import numpy as np

N = 30
M = 20
P = np.random.rand(N, 3)
Q = np.random.rand(M, 3)

# --- METHOD A: The Protocol Way (Subtract, then einsum sum) ---
# einsum cannot do subtraction, so we use your Step 5 alignment to get the diff.
diff = P[:, None, :] - Q[None, :, :]
sq_diff = diff**2

# Now use einsum just to sum out the spatial coordinates 'k' (Axis 2)
# 'ijk' goes in, 'ij' comes out.
D_method_A = np.einsum('ijk -> ij', sq_diff)


# --- METHOD B: The "Machine Learning" einsum Way (High Performance) ---
# Math Trick: (P - Q)^2 = P^2 - 2PQ + Q^2
# This allows us to avoid building the massive (N, M, 3) diff array entirely!

# 1. Sum of P^2 over k. Shape (N, 3) -> (N,) -> Reshape to (N, 1)
P_sq = np.einsum('ik, ik -> i', P, P)[:, None]

# 2. Dot product of P and Q over k. Shape (N, 3) and (M, 3) -> (N, M)
PQ = np.einsum('ik, jk -> ij', P, Q)

# 3. Sum of Q^2 over k. Shape (M, 3) -> (M,) -> Reshape to (1, M)
Q_sq = np.einsum('jk, jk -> j', Q, Q)[None, :]

# Combine using standard broadcasting over the 2D grid
D_method_B = P_sq - 2*PQ + Q_sq

print(f"Shape of Distance Matrix: {D_method_B.shape}")

Shape of Distance Matrix: (30, 20)


# **No Summing Only Multiplication**

However, **np.einsum** does NOT force you to sum! It can do pure element-wise multiplication and broadcasting without any contraction. This is one of its hidden superpowers.

The golden rule of np.einsum is: **If a letter appears on the left side of the -> but NOT on the right side, it gets summed. If it appears on the right side, it is kept!**

Here is how einsum can perfectly replicate your Vectorization Dojo techniques without summing:

### A. Pure Element-Wise Multiplication (No Summation)
If you have two vectors A and B of size $N$, and you just want $C_i = A_i \cdot B_i$:

In [11]:
# The Dojo way:
A =np.random.randint(5,10,size=10)
print(A)
B= np.random.randint(5,10,size =10)
print(B)
C= A*B
print(C)

# The einsum way (notice 'i' is on the right, so it is NOT summed):
C = np.einsum('i, i -> i', A, B)
print(C)

[7 6 7 8 5 7 7 5 7 7]
[9 6 8 8 8 7 7 8 6 7]
[63 36 56 64 40 49 49 40 42 49]
[63 36 56 64 40 49 49 40 42 49]


### B. The Explicit Broadcast / Outer Product (No Summation)
Let's look at the Numerator from your Challenge 1, where you needed an $(N, M)$ grid from two vectors $A$ and $B$.

In [14]:
N = 5
M = 3
A = np.random.rand(N) # Shape (N,)
B = np.random.rand(M) # Shape (M,)

# The Dojo way:
interaction = A[:, None] * B[None, :]
print(interaction)
print(interaction.shape)

# The einsum way (neither 'i' nor 'j' are omitted on the right, so NO sum happens):
interaction = np.einsum('i, j -> ij', A, B)
print(interaction)
print(interaction.shape)

[[0.65750611 0.16744529 0.0454838 ]
 [0.5035805  0.12824547 0.03483581]
 [0.64215595 0.1635361  0.04442194]
 [0.51251463 0.1305207  0.03545384]
 [0.29781569 0.07584391 0.02060177]]
(5, 3)
[[0.65750611 0.16744529 0.0454838 ]
 [0.5035805  0.12824547 0.03483581]
 [0.64215595 0.1635361  0.04442194]
 [0.51251463 0.1305207  0.03545384]
 [0.29781569 0.07584391 0.02060177]]
(5, 3)


### C. Broadcasting a 1D array against a 2D matrix (No Summation)
Imagine multiplying every column of a 2D matrix W by a 1D vector X element-wise, but not summing them up.

In [ ]:
# The Dojo way:
T = W[:, :] * X[None, :]

# The einsum way:
T = np.einsum('ij, j -> ij', W, X)

## **Limitations of einsum**

Even though einsum can do element-wise multiplication and broadcasting without summing, **your Dojo protocol is still the only reliable way to handle the rest of mathematics. np.einsum strictly only does multiplication** .

It cannot do:

**Subtraction: P[:, None, :] - Q[None, :, :]**

**Division: numerator / denominator[:, None]**

**Addition: A[:, None] + B[None, :]**


Because of this, your Step 5 (The Alignment using None) is still completely indispensable for building robust, vectorized code. einsum is just a specialized tool you can pull out of your Dojo toolkit specifically when things need to be multiplied and/or summed!